# Projet Home Credit — Sujet 03

*Séance :* 02  
*Nom :* HOUNYO Charbel  
*Porteur :* P2

**Étapes assignées :**

- 2.2 : mesurer la dispersion du score en répétant le même protocole avec plusieurs tirages.
- 2.5 : entraîner un seul modèle, sans réglage, et noter ses variables et son score sur le contrôle.

*Une aide par IA a été utilisée pour préparer et vérifier ces cellules ; déclare-la selon les consignes et assure-toi de pouvoir expliquer chaque choix.*

# Carnet de démarrage — sujet 03 · Home Credit

**Objectif du sujet** : décider d'accorder ou de refuser un crédit.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

8,07 % de défauts. Les manquants ne sont pas comblés : c'est une information.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/17v1qEMO8FohlDPU99fAjEcuy_nl-Nim5?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_03_Home_Credit/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |
| `HomeCredit_columns_description.csv` | entraînement ou annexe |

**Format de sortie attendu** : `SK_ID_CURR,TARGET`.
**Colonne d'identifiant** : `SK_ID_CURR`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


In [1]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [2]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv', 'HomeCredit_columns_description.csv']
SUJET = "Sujet_03_Home_Credit"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : C:\Users\HP\Downloads\HomeCredit
fichiers vus : ['Carnet_de_demarrage.ipynb', 'Enonce.md', 'HomeCredit_columns_description.csv', 'a_decider.csv', 'soumission.csv', 'train.csv']


In [3]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()


In [4]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (246008, 122)
colonnes : 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,123551,0,Cash loans,M,N,Y,0,202500.0,1078200.0,31653.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,400429,0,Cash loans,F,N,Y,2,148500.0,544491.0,20133.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0
2,106912,0,Cash loans,F,Y,N,2,67500.0,363190.5,24268.5,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,3.0
3,383991,0,Revolving loans,F,N,Y,1,144000.0,450000.0,22500.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
4,423763,0,Cash loans,F,N,Y,0,67500.0,675000.0,21775.5,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0


In [9]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "TARGET"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    246008.000000
mean          0.080729
std           0.272419
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: TARGET, dtype: float64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [10]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['SK_ID_CURR', 'TARGET']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"SK_ID_CURR": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["TARGET"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["SK_ID_CURR"], complet=True)
identifiants = a_predire["SK_ID_CURR"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 61503
écrit : soumission.csv · 61,503 lignes · colonnes ['SK_ID_CURR', 'TARGET']


,SK_ID_CURR,TARGET
0,424613,None
1,260084,None
2,214262,None
3,357905,None
4,101573,None
...,...,...
61498,342357,None
61499,382938,None
61500,235122,None
61501,348296,None


## Étape 2.2 — mesurer le bruit du score

Le document demande de répéter la même mesure en ne changeant que le tirage, puis de quantifier la dispersion du score dans l’unité de la métrique. Ici, on compare cinq découpages stratifiés reproductibles (80 % apprentissage, 20 % contrôle). La préparation des variables et le modèle restent identiques à chaque répétition ; chaque imputation et encodage est ajusté uniquement sur la partie apprentissage du tirage concerné.

La métrique de l’équipe n’étant pas encore arrêtée, le carnet affiche l’AUC (indépendante d’un seuil) ainsi que la précision, le rappel et l’exactitude au seuil fixe de 0,50. Pour chacune, il donne la moyenne, l’écart-type et l’étendue observée. La précision et le rappel sont ici des repères au seuil 0,50, pas un seuil optimisé. Les cinq découpages servent à mesurer la sensibilité au tirage, pas à régler le modèle.

In [ ]:
# --- Étape 2.2 : répéter la mesure en ne changeant que le tirage --------------
import gc
import time

import numpy as np
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Libère l'aperçu chargé plus haut avant de lire les données complètes.
if "entrainement" in globals():
    del entrainement
gc.collect()

print("Chargement complet de train.csv…", flush=True)
train_complet = table("train.csv", complet=True)
CIBLE = "TARGET"
IDENTIFIANT = "SK_ID_CURR"

# L'identifiant n'est pas une information métier ; la cible ne doit jamais être un prédicteur.
colonnes_variables = [c for c in train_complet.columns if c not in [CIBLE, IDENTIFIANT]]
X = train_complet[colonnes_variables]
y = train_complet[CIBLE].astype("int8")

colonnes_numeriques = X.select_dtypes(include=["number"]).columns.tolist()
colonnes_categorielles = [c for c in colonnes_variables if c not in colonnes_numeriques]

preparation = ColumnTransformer(
    transformers=[
        (
            "numeriques",
            Pipeline([
                ("imputation", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                ("standardisation", StandardScaler()),
            ]),
            colonnes_numeriques,
        ),
        (
            "categorielles",
            Pipeline([
                ("imputation", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                ("encodage", OneHotEncoder(handle_unknown="ignore")),
            ]),
            colonnes_categorielles,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

# Un seul algorithme, sans recherche d'hyperparamètres : régression logistique standard.
modele_reference = Pipeline([
    ("preparation", preparation),
    ("classification", LogisticRegression(max_iter=300)),
])

GRAINES_TIRAGE = [42, 52, 62, 72, 82]
TAILLE_CONTROLE = 0.20
SEUIL_COMPLEMENTAIRE = 0.50
indices = np.arange(len(y))
resultats_tirages = []
modele_controle = None
scores_controle = None

print(
    f"Données : {len(y):,} demandes ; {len(colonnes_variables)} variables "
    f"({len(colonnes_numeriques)} numériques, {len(colonnes_categorielles)} catégorielles).",
    flush=True,
)

for graine in GRAINES_TIRAGE:
    indices_apprentissage, indices_controle = train_test_split(
        indices,
        test_size=TAILLE_CONTROLE,
        random_state=graine,
        stratify=y,
    )
    y_apprentissage = y.iloc[indices_apprentissage]
    y_controle = y.iloc[indices_controle]
    X_apprentissage = X.iloc[indices_apprentissage]
    X_controle = X.iloc[indices_controle]

    modele_tirage = clone(modele_reference)
    debut = time.perf_counter()
    modele_tirage.fit(X_apprentissage, y_apprentissage)
    probabilites = modele_tirage.predict_proba(X_controle)[:, 1]
    predictions = (probabilites >= SEUIL_COMPLEMENTAIRE).astype("int8")
    duree = time.perf_counter() - debut

    scores = {
        "AUC": roc_auc_score(y_controle, probabilites),
        "Précision @ 0,50": precision_score(y_controle, predictions, zero_division=0),
        "Rappel @ 0,50": recall_score(y_controle, predictions, zero_division=0),
        "Exactitude @ 0,50": accuracy_score(y_controle, predictions),
    }
    resultats_tirages.append({"graine": graine, **scores, "secondes": duree})
    print(f"Tirage {graine} terminé en {duree:.1f} s.", flush=True)

    if graine == GRAINES_TIRAGE[0]:
        # Le tirage 42 devient le contrôle fixe présenté à l'étape 2.5.
        modele_controle = modele_tirage
        scores_controle = scores
        lignes_apprentissage_controle = len(indices_apprentissage)
        lignes_controle = len(indices_controle)
        iterations_modele_controle = int(modele_controle.named_steps["classification"].n_iter_[0])
    else:
        del modele_tirage

    del X_apprentissage, X_controle, y_apprentissage, y_controle
    del probabilites, predictions, scores, indices_apprentissage, indices_controle
    gc.collect()

resultats_mesure = pd.DataFrame(resultats_tirages).set_index("graine")
print("\nScores par tirage :")
print(resultats_mesure.drop(columns="secondes").to_string(float_format=lambda valeur: f"{valeur:.4f}"))

for metrique in ["AUC", "Précision @ 0,50", "Rappel @ 0,50", "Exactitude @ 0,50"]:
    valeurs = resultats_mesure[metrique].to_numpy()
    moyenne = float(np.mean(valeurs))
    dispersion = float(np.std(valeurs, ddof=1))
    minimum, maximum = float(np.min(valeurs)), float(np.max(valeurs))
    if metrique == "AUC":
        print(
            f"{metrique} — moyenne={moyenne:.4f}, écart-type={dispersion:.4f}, "
            f"min={minimum:.4f}, max={maximum:.4f}, étendue={maximum - minimum:.4f}"
        )
    else:
        print(
            f"{metrique} — moyenne={moyenne:.2%}, écart-type={dispersion:.2%} "
            f"(points de pourcentage), min={minimum:.2%}, max={maximum:.2%}, "
            f"étendue={maximum - minimum:.2%}"
        )

# Le modèle du premier tirage est gardé pour documenter l'étape 2.5.
del train_complet, X, y, indices, modele_reference, preparation
gc.collect()

## Étape 2.5 — faire tourner un seul modèle

On retient un seul modèle non réglé : une régression logistique avec ses paramètres standards (aucune recherche d’hyperparamètres). Toutes les variables de `train.csv` sont utilisées sauf `SK_ID_CURR` (identifiant) et `TARGET` (la réponse à prédire). Les variables numériques sont imputées par leur médiane puis standardisées ; les catégorielles sont imputées par la modalité la plus fréquente puis encodées en indicatrices. Ces opérations sont apprises sur les seules lignes d’apprentissage.

Le contrôle fixe est le premier découpage stratifié (graine 42, 80/20) déjà mesuré à l’étape 2.2 : le score affiché ci-dessous est donc celui d’un unique ajustement de ce modèle, sans réentraînement ni réglage supplémentaire. L’AUC et les métriques au seuil fixe 0,50 sont rapportées afin que l’équipe puisse retenir sa métrique commune.

In [ ]:
# --- Étape 2.5 : documenter le modèle unique et son contrôle ------------------
if modele_controle is None or scores_controle is None:
    raise RuntimeError("Exécute d'abord l'étape 2.2 pour créer le modèle et le contrôle fixes.")

print("Modèle : régression logistique (scikit-learn LogisticRegression, paramètres non réglés).")
print("Prétraitement : médiane + indicateurs de manquants et standardisation (numériques) ;")
print("              modalité la plus fréquente + one-hot (catégorielles).")
print(f"Découpage de contrôle : graine {GRAINES_TIRAGE[0]}, stratifié, {lignes_apprentissage_controle:,} apprentissage / {lignes_controle:,} contrôle.")
print(f"Variables fournies ({len(colonnes_variables)}) :")
print(", ".join(colonnes_variables))
print("\nScore sur le contrôle :")
print(f"AUC : {scores_controle['AUC']:.4f}")
print(f"Précision au seuil 0,50 : {scores_controle['Précision @ 0,50']:.2%}")
print(f"Rappel au seuil 0,50 : {scores_controle['Rappel @ 0,50']:.2%}")
print(f"Exactitude au seuil 0,50 : {scores_controle['Exactitude @ 0,50']:.2%}")
print(f"Itérations effectuées : {iterations_modele_controle}")

### Résultats obtenus sur les fichiers actuels

Exécution locale de `train.csv` (246 008 lignes), cinq validations stratifiées 80/20, même régression logistique et même prétraitement. Le contrôle fixe est le tirage 42. Ces résultats correspondent aux fichiers actuels ; relancer l’étape 2.2 si les données ou le protocole changent.

| Graine | AUC | Précision @ 0,50 | Rappel @ 0,50 | Exactitude @ 0,50 |
|---:|---:|---:|---:|---:|
| 42 | 0,7475 | 50,00 % | 1,13 % | 91,93 % |
| 52 | 0,7390 | 48,19 % | 1,01 % | 91,92 % |
| 62 | 0,7440 | 58,95 % | 1,41 % | 91,96 % |
| 72 | 0,7489 | 60,29 % | 1,03 % | 91,96 % |
| 82 | 0,7413 | 50,48 % | 1,33 % | 91,93 % |

- AUC : moyenne **0,7441**, écart-type **0,0041**, étendue **0,0099** (min 0,7390 ; max 0,7489).
- Précision au seuil 0,50 : moyenne **53,58 %**, écart-type **5,60 points**.
- Rappel au seuil 0,50 : moyenne **1,18 %**, écart-type **0,18 point**.
- Exactitude au seuil 0,50 : moyenne **91,94 %**, écart-type **0,02 point**.
- Sur le contrôle fixe (graine 42), le modèle obtient une AUC de **0,7475**. Au seuil 0,50, il a une précision de **50,00 %** et un rappel de **1,13 %**.

Le rappel très faible au seuil 0,50 signifie que ce modèle laisse passer la plupart des défauts à ce seuil. La précision et le rappel changeraient avec un autre seuil ; aucun seuil n’a été optimisé ici. La métrique à retenir reste à confirmer avec l’équipe. Les cinq ajustements ont duré environ **27 secondes chacun** sur cet ordinateur.